In [8]:
from google.colab import drive
import os

# Mount Google Drive to access your dataset
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
import pandas as pd
import numpy as np
import shutil
from tqdm.auto import tqdm
from pathlib import Path

# Set base paths
BASE_DIR = Path('/content/drive/MyDrive/stock_dataset')
CSV_PATH = BASE_DIR / 'labels.csv'
OUTPUT_DIR = BASE_DIR / 'dataset'

### Load and Sort Dataset
We load the CSV, convert timestamps to datetime objects, and sort chronologically to ensure the split respects the time series nature of stock data.

In [10]:
def load_and_sort_data(csv_path):
    """Loads CSV and sorts by timestamp for chronological splitting."""
    df = pd.read_csv(csv_path)
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.sort_values('timestamp').reset_index(drop=True)
    df['label'] = df['label'].str.lower()
    return df

df = load_and_sort_data(CSV_PATH)
display(df.head())

,image_path,label,stock,timestamp
0,/content/drive/MyDrive/stock_dataset/raw_image...,down,RELIANCE,2026-01-07 14:15:00+05:30
1,/content/drive/MyDrive/stock_dataset/raw_image...,neutral,LTTS,2026-01-07 14:15:00+05:30
2,/content/drive/MyDrive/stock_dataset/raw_image...,down,DATAPATTNS,2026-01-07 14:15:00+05:30
3,/content/drive/MyDrive/stock_dataset/raw_image...,down,VOLTAS,2026-01-07 14:15:00+05:30
4,/content/drive/MyDrive/stock_dataset/raw_image...,down,TATAELXSI,2026-01-07 14:15:00+05:30


### Define Split Indices
We calculate the split points for 70% Train, 20% Validation, and 10% Test without shuffling.

In [11]:
def split_per_stock(df):
    train_list, val_list, test_list = [], [], []

    for stock in df['stock'].unique():
        stock_df = df[df['stock'] == stock].sort_values('timestamp')

        n = len(stock_df)
        train_end = int(n * 0.7)
        val_end = int(n * 0.9)

        train_list.append(stock_df.iloc[:train_end])
        val_list.append(stock_df.iloc[train_end:val_end])
        test_list.append(stock_df.iloc[val_end:])

    train_df = pd.concat(train_list).reset_index(drop=True)
    val_df = pd.concat(val_list).reset_index(drop=True)
    test_df = pd.concat(test_list).reset_index(drop=True)

    return train_df, val_df, test_df

train_df, val_df, test_df = split_per_stock(df)

### Create Directory Structure
This step sets up the folders: `train/up`, `train/down`, etc.

In [12]:
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

In [13]:
def setup_directories(base_output, splits, labels):
    """Creates the nested directory structure on Drive."""
    for split in splits:
        for label in labels:
            path = base_output / split / str(label).lower()
            path.mkdir(parents=True, exist_ok=True)

labels = ['up', 'down', 'neutral']
setup_directories(OUTPUT_DIR, ['train', 'val', 'test'], labels)

### Copy Images
We iterate through the split DataFrames and copy files to their new destinations, skipping missing ones.

In [14]:
from shutil import copy2

def distribute_images(df, split_name, base_output):
    print(f"Processing {split_name} split...")

    for row in tqdm(df.itertuples(), total=len(df)):
        src = Path(row.image_path)
        if not src.exists():
            continue

        dest = base_output / split_name / row.label.lower() / src.name

        if not dest.exists():
            copy2(src, dest)

distribute_images(train_df, 'train', OUTPUT_DIR)
distribute_images(val_df, 'val', OUTPUT_DIR)
distribute_images(test_df, 'test', OUTPUT_DIR)

Processing train split...


  0%|          | 0/13192 [00:00<?, ?it/s]

Processing val split...


  0%|          | 0/3770 [00:00<?, ?it/s]

Processing test split...


  0%|          | 0/1905 [00:00<?, ?it/s]

### Final Summary
Displaying the sample counts and class distributions for each split.

In [15]:
def print_summary(splits_dict):
    """Prints dataset statistics and class distribution."""
    for name, df_split in splits_dict.items():
        print(f"\n--- {name.upper()} SPLIT ---")
        print(f"Total Samples: {len(df_split)}")
        print("Class Distribution:")
        print(df_split['label'].value_counts(normalize=True) * 100)

print_summary({'train': train_df, 'val': val_df, 'test': test_df})


--- TRAIN SPLIT ---
Total Samples: 13192
Class Distribution:
label
down       44.534566
up         33.050334
neutral    22.415100
Name: proportion, dtype: float64

--- VAL SPLIT ---
Total Samples: 3770
Class Distribution:
label
down       52.360743
up         33.766578
neutral    13.872679
Name: proportion, dtype: float64

--- TEST SPLIT ---
Total Samples: 1905
Class Distribution:
label
down       46.194226
up         43.307087
neutral    10.498688
Name: proportion, dtype: float64
